# 🧭 The Deterministic Horizon — Quickstart

*Official code for the Deterministic Horizon paper.* This notebook reproduces the headline finding **offline, with no API keys**, in under a minute:

1. Generate BFS-optimal permutation puzzles at increasing depth.
2. Simulate a neural chain-of-thought reasoner whose per-step error follows the paper's model $\varepsilon(d)=\varepsilon_0+\gamma\,d/L_{\text{eff}}$ (Theorem 4.2).
3. Fit the super-exponential decay and locate the **Deterministic Horizon** $d^*$.
4. Route a toy agent with `should_delegate`.

> Prefer to *drag the parameters*? Open the [interactive explorer](https://bettyguo.github.io/deterministic-horizon/).

---

In [ ]:
# On Colab, install the package from GitHub (skipped automatically if already installed).
try:
    import deterministic_horizon  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/bettyguo/deterministic-horizon"

In [ ]:
import random

import numpy as np
from deterministic_horizon import PermutationTask, estimate_horizon

# Paper-canonical decoherence constants (Theorem 4.2): place d* ~ 22.
SEED, N_ELEMENTS = 42, 8
DEPTHS = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28]  # all <= C(8,2)=28
INSTANCES_PER_DEPTH = 120
EPS0, GAMMA, L_EFF = 0.02, 0.15, 150


def simulate_neural_cot(task, instance, rng):
    """Corrupt step i with probability eps(i)=eps0 + gamma*i/L_eff (Thm 4.2)."""
    state = list(instance.initial_state)
    for step, optimal_op in enumerate(instance.optimal_solution):
        eps = min(EPS0 + GAMMA * step / L_EFF, 0.95)
        if rng.random() < eps:
            op = rng.choice([o for o in task.operators if o != optimal_op])
        else:
            op = optimal_op
        state = task.apply_operator(state, op)
    return task.state_equal(state, instance.target_state)

In [ ]:
rng = random.Random(SEED)
task = PermutationTask(seed=SEED, n_elements=N_ELEMENTS)

results, table = [], []
for depth in DEPTHS:
    hits = 0
    for _ in range(INSTANCES_PER_DEPTH):
        inst = task.generate_instance(target_depth=depth)
        correct = simulate_neural_cot(task, inst, rng)
        hits += int(correct)
        results.append({"condition": "C1", "optimal_depth": depth, "correct": correct})
        results.append({"condition": "C3", "optimal_depth": depth, "correct": True})  # exact BFS tool
    table.append((depth, 100 * hits / INSTANCES_PER_DEPTH))

print(f"{'depth':>5}  {'neural CoT':>11}  {'tool (BFS)':>11}")
for depth, pct in table:
    print(f"{depth:>5}  {pct:>10.1f}%  {100.0:>10.1f}%")

In [ ]:
cot = [r for r in results if r["condition"] == "C1"]
horizon = estimate_horizon(cot, threshold=0.5)
print(f"Deterministic Horizon  d* = {horizon['d_star']:.1f}")
print(f"decoherence-model fit  R\u00b2 = {horizon['r_squared']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

depths = np.array([d for d, _ in table])
acc = np.array([p / 100 for _, p in table])
d_star = horizon["d_star"]
eps0, gamma = horizon.get("eps0", EPS0), horizon.get("gamma", GAMMA)
grid = np.linspace(0, 50, 200)
theory = np.exp(-eps0 * grid - gamma * grid * (grid + 1) / 2.0)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(depths, acc, "o", color="#d6332e", ms=7, label="Neural CoT (empirical)")
ax.plot(grid, theory, "-", color="#1f5fb4", lw=2.2, label="Theory (Thm 4.2)")
ax.axhline(0.92, ls="--", color="#2a9d2a", lw=1.8, label="Tool-integrated (C3) ≈ 92%")
ax.axvline(d_star, ls=":", color="#555")
ax.text(d_star + 0.5, 0.05, f"$d^*={d_star:.0f}$", color="#333")
ax.axhline(0.5, color="#aaa", lw=0.8)
ax.set_xlabel("reasoning depth  d")
ax.set_ylabel("accuracy")
ax.set_xlim(0, 50)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.set_title("Accuracy decays super-exponentially; the wall is d*")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Route your own agent

The whole paper collapses to one decision: *given this subproblem's estimated depth, think or delegate?*

In [ ]:
from deterministic_horizon import delegation_decision, horizon_for, should_delegate

for model in ["gpt-4o", "claude-4.5-opus", "o3-mini"]:
    print(f"{model:>16}:  d* = {horizon_for(model):.0f}")

print()
print("depth=8  ->", should_delegate(estimated_depth=8, model="gpt-4o"))    # think
print("depth=35 ->", should_delegate(estimated_depth=35, model="gpt-4o"))   # delegate
print()
print(delegation_decision(estimated_depth=30, model="claude-4.5-opus").explain())

## Where to next

- **[When to delegate](https://github.com/bettyguo/deterministic-horizon/blob/main/docs/when-to-delegate.md)** — per-model horizons + three production routing patterns.
- **[Theorem cheat-sheet](https://github.com/bettyguo/deterministic-horizon/blob/main/docs/theorem-cheatsheet.md)** — every theorem in plain English.
- **[Interactive explorer](https://bettyguo.github.io/deterministic-horizon/)** — drag $\varepsilon_0,\gamma,L_{\text{eff}},\alpha$ and watch $d^*$ move.
- Run on **real LLMs** with the `dh evaluate` CLI (needs API keys; see the README).